In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.delhi_air.gold_station_hourly AS
SELECT s.hour_ist, s.sensor_id, s.location_id, s.location_name, s.provider,
       s.pm25_avg, s.readings, s.expected_readings,
       s.readings >= s.expected_readings AS is_complete_hour,
       m.model_pm25_ugm3,
       w.temperature_c, w.humidity_pct, w.wind_speed_kmh, w.wind_direction_deg,
       w.pressure_hpa, w.precipitation_mm, w.boundary_layer_m
FROM workspace.delhi_air.silver_pm25_hourly s
LEFT JOIN workspace.delhi_air.silver_model_pm25_hourly m ON s.hour_ist = m.hour_ist
LEFT JOIN workspace.delhi_air.silver_weather_hourly w ON s.hour_ist = w.hour_ist
""")

spark.sql("""
    SELECT count(*) AS total_rows,
           sum(CASE WHEN model_pm25_ugm3 IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_model,
           sum(CASE WHEN is_complete_hour THEN 1 ELSE 0 END) AS complete_hours,
           min(hour_ist) AS first_hour, max(hour_ist) AS last_hour
    FROM workspace.delhi_air.gold_station_hourly
""").show(truncate=False)